# Explainability: Burnout Risk Prediction (Classification)

In [1]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import StandardScaler, LabelEncoder
from pytorch_tabnet.tab_model import TabNetClassifier
warnings.filterwarnings('ignore')
shap.initjs()

# 1. Load Data
df = pd.read_csv('data/developer_burnout_dataset_7000.csv')
df.dropna(subset=['stress_level', 'burnout_level'], inplace=True)
for col in df.select_dtypes(include=[np.number]).columns:
    df[col].fillna(df[col].median(), inplace=True)
    
X = df.drop(columns=['stress_level', 'burnout_level'])
le = LabelEncoder()
y = le.fit_transform(df['burnout_level'])
feature_names = X.columns

# 2. Preprocessing
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_names)

ModuleNotFoundError: No module named 'shap'

## 1. Random Forest Classifier

In [ ]:
rf_c_model = joblib.load('random_forest/random_forest_classifier_optimized.pkl')

plt.figure(figsize=(10, 6))
sns.barplot(x=rf_c_model.feature_importances_, y=feature_names, palette="viridis")
plt.title("RF Classification - Built-in Feature Importances")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

# SHAP Values for Random Forest
explainer_rfc = shap.TreeExplainer(rf_c_model)
shap_values_rfc = explainer_rfc.shap_values(X_scaled_df)
shap.summary_plot(shap_values_rfc, X_scaled_df, plot_type="bar")

## 2. Support Vector Classifier (SVC)

In [ ]:
svc_model = joblib.load('SVM/svc_burnout_model_optimized.pkl')

X_sample_c = shap.sample(X_scaled_df, 50)
explainer_svc = shap.KernelExplainer(svc_model.predict_proba, X_sample_c)
shap_values_svc = explainer_svc.shap_values(X_scaled_df.iloc[:200])

# Force plot or summary per class 
shap.summary_plot(shap_values_svc, X_scaled_df.iloc[:200], class_names=le.classes_)

## 3. TabNet Classifier

In [ ]:
tabnet_c_model = TabNetClassifier()
tabnet_c_model.load_model('TabNet/tabnet_burnout_model_optimized.zip')

explain_matrix, masks = tabnet_c_model.explain(X_scaled_df.values.astype(np.float32))
feature_importances = explain_matrix.mean(axis=0)

plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances, y=feature_names, palette="magma")
plt.title("TabNet Classification - Feature Importances (via mask explain)")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

## 4. KMeans Clustering

In [ ]:
kmeans_model = joblib.load('kmeans/kmeans_model.pkl')
kmeans_scaler = joblib.load('kmeans/kmeans_scaler.pkl')
cluster_probs = joblib.load('kmeans/kmeans_burnout_cluster_probs.pkl')

centroids = kmeans_scaler.inverse_transform(kmeans_model.cluster_centers_)
clusters_df = pd.DataFrame(centroids, columns=feature_names)
clusters_df.index.name = "Cluster"

plt.figure(figsize=(12, 6))
sns.heatmap(clusters_df, annot=True, cmap="coolwarm", fmt=".1f", linewidths=.5)
plt.title("KMeans Centroids Overview (Feature impact per cluster)")
plt.show()